# Proyecto Final · Bibliotecas públicas de Barcelona
## Notebook 01 —  FASE 01: Extracción de datos (Open Data BCN)

**Objetivo:** descargar los datos crudos desde la API CKAN de Open Data BCN y
guardarlos sin transformar en `data/raw/`.

Fuentes:
1. Bibliotecas — préstamo presencial (`dades-xarxa-biblioteques-catalunya`)
2. Padrón — demografía por distrito (`pad_*`)
3. Renta — renta disponible per cápita (`renda-disponible-llars-bcn`)

En este notebook **solo extraemos** (descargar + guardar). La limpieza va en el notebook 02.

celda 1

In [ ]:
#celda 2

import requests
import pandas as pd
from pathlib import Path

# Endpoint base de la API CKAN de Open Data BCN
BASE_URL = "https://opendata-ajuntament.barcelona.cat/data/api/3/action"

# Carpeta donde guardaremos los datos crudos (se crea si no existe)
RAW = Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)

def get_action(action, params=None):
    """Llama a una acción de la API CKAN y devuelve el contenido de 'result'.

    action: nombre de la acción, p. ej. 'package_show' o 'datastore_search'
    params: diccionario con los parámetros de la petición
    """
    url = f"{BASE_URL}/{action}"
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()          # lanza error si la respuesta no es 200 OK
    data = resp.json()               # convierte el JSON en un dict de Python
    if not data.get("success"):      # CKAN indica si la petición fue válida
        raise RuntimeError(f"La API devolvió success=False: {data}")
    return data["result"]            # nos quedamos solo con el contenido útil

### Paso 1: explorar el dataset de bibliotecas

Antes de descargar nada, pedimos la "ficha" del dataset con `package_show`.
Nos interesa su lista de **recursos** (`resources`): hay un fichero por año
(2010–2024). De cada recurso necesitaremos su `id` (el `resource_id`) y si está
cargado en el *datastore* (consultable vía API) o solo como CSV.

celda 3

In [ ]:
#celda 4

# Pedimos la ficha del dataset de la red de bibliotecas
paquete = get_action("package_show", {"id": "dades-xarxa-biblioteques-catalunya"})

# 'resources' es una lista de diccionarios, uno por fichero. La pasamos a DataFrame
recursos = pd.DataFrame(paquete["resources"])

# Mostramos solo las columnas que nos importan para decidir cómo bajar cada año
print("Nº de recursos:", len(recursos))
recursos[["name", "format", "datastore_active", "id"]]

Nº de recursos: 30


,name,format,datastore_active,id
0,2024_dades_biblioteques.csv,CSV,False,ce02d3d8-eab4-46ec-8a9a-1afd72f11489
1,2024_dades_biblioteques.xml,XML,False,dba00beb-d835-4f11-8f54-b8edaf5c6057
2,2023_dades_biblioteques.csv,CSV,False,7409382f-45ae-443a-ae70-3653526f3859
3,2023_dades_biblioteques.xml,XML,False,6ccf62aa-6c8f-49b3-8296-a756fe9e71d7
4,2022_dades_biblioteques.csv,CSV,True,8acc3cb1-8014-48fc-b367-ac13a8745061
5,2022_dades_biblioteques.xml,XML,False,b9192162-01e8-4c47-a853-eb3fadc28d67
6,2021_dades_biblioteques.csv,CSV,True,1a83e7d4-80b2-4b98-a253-8ded5426e657
7,2021_dades_biblioteques.xml,XML,False,6a8b829d-a93a-4e84-ac84-acc4804ab2b5
8,2020_dades_biblioteques.csv,CSV,True,80bbaeeb-a301-4017-be08-a1be33f0e276
9,2020_dades_biblioteques.xml,XML,False,ee86eb3b-54d8-4e0e-a4f1-f25831aace59


### Paso 2: seleccionar los recursos que vamos a descargar

De los 30 recursos nos quedamos solo con los **CSV** (uno por año) y descartamos los XML.
Después los separamos en dos grupos según `datastore_active`:
- `True`  → 2010–2022 → se descargan vía API (`datastore_search`)
- `False` → 2023–2024 → se leen como CSV directo desde su URL

Extraemos también el **año** desde el nombre del fichero para poder ordenar y etiquetar.

celda 5

In [ ]:
#celda 6

# 1) Nos quedamos solo con los recursos en formato CSV (descartamos los XML)
csv = recursos[recursos["format"] == "CSV"].copy()

# 2) Sacamos el año del nombre del fichero (p. ej. "2022_dades_biblioteques.csv" -> 2022)
#    str.extract con una expresión regular: \d{4} = cuatro dígitos seguidos
csv["any"] = csv["name"].str.extract(r"(\d{4})").astype(int)

# 3) Ordenamos por año para que quede prolijo
csv = csv.sort_values("any").reset_index(drop=True)

# 4) Separamos en dos grupos según dónde viven los datos
via_api = csv[csv["datastore_active"] == True]    # 2010–2022
via_csv = csv[csv["datastore_active"] == False]   # 2023–2024

print("Total CSV (uno por año):", len(csv))
print("Por API (datastore):", via_api["any"].tolist())
print("Por CSV directo:", via_csv["any"].tolist())

csv[["any", "name", "datastore_active", "url", "id"]]

Total CSV (uno por año): 15
Por API (datastore): [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
Por CSV directo: [2023, 2024]


,any,name,datastore_active,url,id
0,2010,2010_dades_biblioteques.csv,True,https://opendata-ajuntament.barcelona.cat/data...,2fb00031-9aa7-459d-ab07-4bf059960fbd
1,2011,2011_dades_biblioteques.csv,True,https://opendata-ajuntament.barcelona.cat/data...,2b02d54f-ea99-4a80-8e44-b89b5dfe79f6
2,2012,2012_dades_biblioteques.csv,True,https://opendata-ajuntament.barcelona.cat/data...,0e3c12f1-88f7-4ada-b4d9-4e7501306766
3,2013,2013_dades_biblioteques.csv,True,https://opendata-ajuntament.barcelona.cat/data...,160d5835-c911-4584-a2c4-c556b9eeb365
4,2014,2014_dades_biblioteques.csv,True,https://opendata-ajuntament.barcelona.cat/data...,446e6e23-2af2-4c75-8014-c768e9d41b5a
5,2015,2015_dades_biblioteques.csv,True,https://opendata-ajuntament.barcelona.cat/data...,a578ae65-4d76-46ab-9d48-a68d645ebad4
6,2016,2016_dades_biblioteques.csv,True,https://opendata-ajuntament.barcelona.cat/data...,27369e70-3932-49d9-ac06-36fe13e85a7d
7,2017,2017_dades_biblioteques.csv,True,https://opendata-ajuntament.barcelona.cat/data...,f3dbe73b-8720-4d4f-9f4e-fb81b219fbc8
8,2018,2018_dades_biblioteques.csv,True,https://opendata-ajuntament.barcelona.cat/data...,96313aff-8b64-4322-94b0-8043a2ae7068
9,2019,2019_dades_biblioteques.csv,True,https://opendata-ajuntament.barcelona.cat/data...,3891e73c-2d25-45b8-b5ee-9178aa5db4ad


### Paso 3: descargar un año por la API (prueba con 2022)

`datastore_search` devuelve los registros de UN recurso. Como pagina (100 filas
por defecto), creamos una función que recorre todas las páginas con `limit` y
`offset` hasta traer el total.

Probamos con 2022 para: (1) ver las columnas reales, y (2) confirmar el nombre
exacto del indicador de préstamo presencial antes de descargar todos los años.

celda 7

In [ ]:
# celda 8

def fetch_datastore(resource_id, page_size=1000):
    """Descarga TODOS los registros de un recurso del datastore, paginando."""
    registros = []
    offset = 0
    while True:
        result = get_action("datastore_search", {
            "resource_id": resource_id,
            "limit": page_size,     # cuántas filas por tanda
            "offset": offset,       # desde qué fila empezar
        })
        tanda = result["records"]
        registros.extend(tanda)             # añadimos las filas de esta tanda
        offset += page_size
        if offset >= result["total"]:       # ya hemos traído todas
            break
    return pd.DataFrame(registros)

# Cogemos el resource_id del año 2022 desde la tabla 'via_api'
id_2022 = via_api.loc[via_api["any"] == 2022, "id"].iloc[0]

df_2022 = fetch_datastore(id_2022)

print("Filas descargadas:", len(df_2022))
print("\nColumnas:", list(df_2022.columns))
print("\nIndicadores disponibles:")
print(df_2022["Indicador"].unique())

Filas descargadas: 400

Columnas: ['Latitud', 'Titularitat', 'Codi_Districte', 'Tipus_Us', 'Notes_Equipament', 'Nom_Districte', 'Notes_Dades', 'Nom_Barri', 'Indicador', 'Tipus_Equipament', 'Nom_Equipament', 'Valor', 'Codi_Barri', 'Ambit', '_id', 'Any', 'Longitud']

Indicadores disponibles:
['Assistents_Activitats' 'Metres_Quadrats' 'Assistents_Visites_Escolars'
 'Fons_Documentals' 'Nombre_Activitats' 'Nombre_Visites_Escolars'
 'Prestecs_presencials' 'Usos_Ordinadors' 'Prestecs_virtuals' 'Visites']


### Paso 4: descargar todos los años del datastore (2010–2022)

Recorremos cada recurso de `via_api`, descargamos sus registros con
`fetch_datastore` y los juntamos en un único DataFrame. Guardamos TODOS los
indicadores (datos crudos); el filtrado a préstamo presencial irá en la Fase 2.

#celda 9

In [5]:
#celda 10

trozos = []   # aquí iremos guardando el DataFrame de cada año

for fila in via_api.itertuples():          # recorremos fila a fila la tabla via_api
    df_anyo = fetch_datastore(fila.id)     # descargamos ese año
    df_anyo["any_recurso"] = fila.any      # etiqueta de control con el año del fichero
    trozos.append(df_anyo)
    print(f"  {fila.any}: {len(df_anyo)} filas")

# Unimos los 13 años en un solo DataFrame
bib_api = pd.concat(trozos, ignore_index=True)

print("\nTotal filas (2010–2022):", len(bib_api))
print("Años incluidos:", sorted(bib_api['any_recurso'].unique()))

  2010: 141 filas
  2011: 148 filas
  2012: 155 filas
  2013: 156 filas
  2014: 320 filas
  2015: 320 filas
  2016: 320 filas
  2017: 320 filas
  2018: 320 filas
  2019: 320 filas
  2020: 400 filas
  2021: 400 filas
  2022: 400 filas

Total filas (2010–2022): 3720
Años incluidos: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


### Paso 5: guardar los datos crudos (2010–2022) en data/raw

Concatenar solo apila los años, no modifica nada: sigue siendo dato crudo.
Lo guardamos ya en `data/raw/` para no depender de volver a llamar a la API.
Los años 2023–2024 (CSV directo) se añadirán en el siguiente paso.

#celda 11

In [6]:
#celda 12

ruta_raw = RAW / "bibliotecas_xarxa_2010_2022_raw.csv"
bib_api.to_csv(ruta_raw, index=False, encoding="utf-8")

print("Guardado en:", ruta_raw.resolve())
print("Filas:", len(bib_api), "| Columnas:", len(bib_api.columns))

Guardado en: C:\Users\User\AppData\Local\Programs\Microsoft VS Code\data\raw\bibliotecas_xarxa_2010_2022_raw.csv
Filas: 3720 | Columnas: 20


### Reubicar el proyecto en el escritorio

El notebook se estaba ejecutando dentro de la carpeta de VS Code. Creamos una
carpeta propia del proyecto en el escritorio y reapuntamos `RAW` allí. Como
`bib_api` sigue en memoria, re-guardamos sin volver a descargar.

celda 13

In [7]:
#celda 14

import os
from pathlib import Path

# Carpeta principal del proyecto, en el ESCRITORIO
PROYECTO = Path.home() / "Desktop" / "proyecto_bibliotecas_bcn"
RAW = PROYECTO / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)   # crea proyecto + data + raw de golpe

# Re-guardamos el raw que ya teníamos en memoria, ahora en el escritorio
bib_api.to_csv(RAW / "bibliotecas_xarxa_2010_2022_raw.csv", index=False, encoding="utf-8")

print("Proyecto creado en:", PROYECTO)
os.startfile(PROYECTO)   # abre la carpeta en el Explorador

Proyecto creado en: C:\Users\User\Desktop\proyecto_bibliotecas_bcn


### Paso 6: descargar 2023–2024 (CSV directo)

Estos años no están en el datastore: los leemos desde su URL con `pd.read_csv`.
Antes de unirlos al bloque 2010–2022, comparamos las columnas para asegurarnos
de que la estructura coincide (verificación de coherencia).

#celda 15

In [8]:
#celda 16

trozos_csv = []
for fila in via_csv.itertuples():          # via_csv tiene 2023 y 2024
    df_anyo = pd.read_csv(fila.url)        # leemos el CSV directo desde su URL
    df_anyo["any_recurso"] = fila.any
    trozos_csv.append(df_anyo)
    print(f"{fila.any}: {df_anyo.shape[0]} filas, {df_anyo.shape[1]} columnas")

bib_csv = pd.concat(trozos_csv, ignore_index=True)

# Verificación: ¿coinciden las columnas con las del bloque del datastore?
cols_api = set(bib_api.columns)
cols_csv = set(bib_csv.columns)
print("\nColumnas solo en el bloque API :", cols_api - cols_csv)
print("Columnas solo en el bloque CSV :", cols_csv - cols_api)

2023: 400 filas, 17 columnas
2024: 400 filas, 17 columnas

Columnas solo en el bloque API : {'Equipament', '_id', 'TipusGeneral'}
Columnas solo en el bloque CSV : set()


### Paso 7: verificar columnas clave y guardar 2023–2024 en raw

No combinamos aún los dos bloques (eso es transformación → Fase 2). Guardamos
2023–2024 como su propio fichero crudo. Antes confirmamos que las columnas que
usaremos están presentes.

#celda 17

In [9]:
#celda 18

# Confirmar que las columnas clave están en el bloque 2023–2024
clave = ["Any", "Indicador", "Valor", "Nom_Equipament", "Codi_Districte", "Nom_Districte"]
print("¿Columnas clave presentes en 2023–2024?")
for col in clave:
    print(f"  {col}: {'sí' if col in bib_csv.columns else 'NO'}")

# Guardar el bloque 2023–2024 como raw (sin tocar)
ruta_csv = RAW / "bibliotecas_xarxa_2023_2024_raw.csv"
bib_csv.to_csv(ruta_csv, index=False, encoding="utf-8")
print("\nGuardado en:", ruta_csv.resolve())
print("Filas:", len(bib_csv), "| Columnas:", len(bib_csv.columns))

¿Columnas clave presentes en 2023–2024?
  Any: sí
  Indicador: sí
  Valor: sí
  Nom_Equipament: sí
  Codi_Districte: sí
  Nom_Districte: sí

Guardado en: C:\Users\User\Desktop\proyecto_bibliotecas_bcn\data\raw\bibliotecas_xarxa_2023_2024_raw.csv
Filas: 800 | Columnas: 17


## Fuente 2: Renta disponible per cápita por distrito

Dataset: `renda-disponible-llars-bcn`. Reutilizamos el mismo patrón:
primero `package_show` para ver qué recursos (años) tiene y si están en el datastore.

celda 19

In [ ]:
#celda 20

paquete_renta = get_action("package_show", {"id": "renda-disponible-llars-bcn"})

recursos_renta = pd.DataFrame(paquete_renta["resources"])
print("Nº de recursos:", len(recursos_renta))
recursos_renta[["name", "format", "datastore_active", "id"]]

Nº de recursos: 8


,name,format,datastore_active,id
0,2022_renda_disponible_llars_per_persona.csv,CSV,True,3df0c5b9-de69-4c94-b924-57540e52932f
1,2021_renda_disponible_llars_per_persona.csv,CSV,True,e14509ca-9cba-43ec-b925-3beb5c69c2c7
2,2020_renda_disponible_llars_per_persona.csv,CSV,True,afe5b67d-7948-4e79-a88c-d51e55fe3ac6
3,2019_renda_disponible_llars_per_persona.csv,CSV,True,0e205580-6d55-4599-bd13-086de83130b8
4,2018_renda_disponible_llars_per_persona.csv,CSV,True,9f5a6152-0075-4111-abec-7b5d62655dd3
5,2017_renda_disponible_llars_per_persona.csv,CSV,True,dc0c1ad4-8b5e-4762-b999-2d4ffc95a718
6,2016_renda_disponible_llars_per_persona.csv,CSV,True,bc48d45b-1046-4496-93c1-fbdac1fd47d0
7,2015_renda_disponible_llars_per_persona.csv,CSV,True,9a69cefe-dcc5-400c-8647-4e438f7ae12b


### Descargar renta (2015–2022) y revisar su estructura

Todos los recursos están en el datastore, así que reutilizamos `fetch_datastore`.
Tras descargar, miramos las columnas para localizar el código de distrito y la
columna del importe de renta por persona.

celda 21

In [11]:
#celda 22

# Año desde el nombre del fichero y ordenamos
recursos_renta["any"] = recursos_renta["name"].str.extract(r"(\d{4})").astype(int)
recursos_renta = recursos_renta.sort_values("any").reset_index(drop=True)

trozos_renta = []
for fila in recursos_renta.itertuples():
    df_anyo = fetch_datastore(fila.id)
    df_anyo["any_recurso"] = fila.any
    trozos_renta.append(df_anyo)
    print(f"  {fila.any}: {len(df_anyo)} filas")

renta = pd.concat(trozos_renta, ignore_index=True)
print("\nTotal filas:", len(renta))
print("Columnas:", list(renta.columns))
renta.head()

  2015: 1068 filas
  2016: 1068 filas
  2017: 1068 filas
  2018: 1068 filas
  2019: 1068 filas
  2020: 1068 filas
  2021: 1068 filas
  2022: 1068 filas

Total filas: 8544
Columnas: ['Codi_Districte', 'Import_Euros', 'Nom_Districte', 'Nom_Barri', 'Seccio_Censal', 'Codi_Barri', '_id', 'Any', 'any_recurso']


,Codi_Districte,Import_Euros,Nom_Districte,Nom_Barri,Seccio_Censal,Codi_Barri,_id,Any,any_recurso
0,1,12845,Ciutat Vella,el Raval,1,1,1,2015,2015
1,1,10442,Ciutat Vella,el Raval,2,1,2,2015,2015
2,1,10048,Ciutat Vella,el Raval,3,1,3,2015,2015
3,1,13121,Ciutat Vella,el Raval,4,1,4,2015,2015
4,1,10579,Ciutat Vella,el Raval,5,1,5,2015,2015


### Guardar renta (2015–2022) en raw

La renta viene a nivel de sección censal (1.068 secciones/año). La guardamos
cruda tal cual. La agregación a distrito (media de Import_Euros) se hará en la
Fase 2, no aquí.

celda 23

In [12]:
#celda 24

ruta_renta = RAW / "renta_disponible_2015_2022_raw.csv"
renta.to_csv(ruta_renta, index=False, encoding="utf-8")

print("Guardado en:", ruta_renta.resolve())
print("Filas:", len(renta), "| Columnas:", len(renta.columns))
print("Años:", sorted(renta["any_recurso"].unique()))
print("Granularidad: sección censal → se agregará a distrito en Fase 2")

Guardado en: C:\Users\User\Desktop\proyecto_bibliotecas_bcn\data\raw\renta_disponible_2015_2022_raw.csv
Filas: 8544 | Columnas: 9
Años: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Granularidad: sección censal → se agregará a distrito en Fase 2


## Fuente 3: Padrón (demografía por distrito)

La familia `pad_*` tiene muchos datasets. Usamos `package_search` (buscar por
palabra clave) en vez de `package_show` (que necesita el id exacto). Buscamos los
conjuntos del padrón y nos quedamos con los nombres reales para elegir
población, edad y origen.

celda 25

In [13]:
#celda 26

# package_search busca datasets por texto; 'rows' = cuántos resultados devolver
busqueda = get_action("package_search", {"q": "padró", "rows": 200})
print("Datasets que coinciden con 'padró':", busqueda["count"])

# Nos quedamos con los que empiezan por 'pad' y mostramos nombre y título
pad_datasets = pd.DataFrame([
    {"name": d["name"], "title": d["title"], "recursos": d.get("num_resources")}
    for d in busqueda["results"]
    if d["name"].startswith("pad")
])

pd.set_option("display.max_colwidth", None)   # para ver los títulos completos
pad_datasets

Datasets que coinciden con 'padró': 60


,name,title,recursos
0,pad_imm_mdbas,Immigrants,58
1,pad_emi_mdbas,Emigrants,58
2,pad_bai_mdbas,Cancellations for undue inscription,58
3,pad_def_mdbas,Deaths,58
4,pad_alt_mdbas,Registrations by omission,58
5,pad_nai_mdbas,Births,58
6,pad_mdb_nacionalitat-regio_sexe,Population by geographical region of nationality and sex,60
7,pad_mdb_lloc-naix-regio_sexe,Population by geographical region of birth and sex,60
8,pad_m_nom_sexe,Names of the inhabitants of Barcelona by average age and sex,60
9,pad_m_cognom,Surnames of the inhabitants of Barcelona,60


### Población: explorar recursos y entender las fechas de referencia

`pad_mdbas` tiene 60 recursos (varias fechas de referencia por año). Antes de
descargar, miramos los nombres para quedarnos con UN recurso por año y no bajar
60 ficheros. También confirmamos que está en el datastore.

celda 27

In [14]:
#celda 28

pkg_pob = get_action("package_show", {"id": "pad_mdbas"})
rec_pob = pd.DataFrame(pkg_pob["resources"])

print("Nº de recursos:", len(rec_pob))
print("Formatos:", rec_pob["format"].unique())
print("¿Datastore activo?:", rec_pob["datastore_active"].unique())

# Vemos los nombres para entender el patrón de fechas
rec_pob[["name", "format", "datastore_active", "id"]].head(30)

Nº de recursos: 60
Formatos: ['CSV' 'JSON']
¿Datastore activo?: [ True False]


,name,format,datastore_active,id
0,2026_pad_mdbas.csv,CSV,True,c7fca94d-91aa-405b-8939-1860aafe7ff5
1,2026_pad_mdbas.json,JSON,False,50c9c892-654b-4ff0-a812-53666082d988
2,2025_pad_mdbas.csv,CSV,True,eb82adf2-a7b0-40e6-9624-b4b9eff23018
3,2025_pad_mdbas.json,JSON,False,9c78eba1-7b4a-4265-bf66-96488f1e24be
4,2024_pad_mdbas.csv,CSV,True,fc597601-a291-4811-ad02-c58e32784692
5,2024_pad_mdbas.json,JSON,False,3cfa8831-a63b-454d-b278-042dc487ed34
6,2023_pad_mdbas.csv,CSV,True,f70020ae-c41a-438d-8983-a372628c197b
7,2023_pad_mdbas.json,JSON,False,6cdb1d7d-d43f-41cf-9070-0d4a91030cd3
8,2022_pad_mdbas.csv,CSV,True,78b965d3-b2dc-4f23-91b9-59caa45bc334
9,2022_pad_mdbas.json,JSON,False,b5a45380-ae78-4a45-bfe9-61b4281d3e16


### Población: seleccionar 2010–2024 y descargar

Patrón igual que bibliotecas: un CSV por año. Filtramos CSV, nos quedamos con
2010–2024 y separamos según `datastore_active` (API vs CSV directo). Reutilizamos
`fetch_datastore` y añadimos lectura directa para los años que no estén en el datastore.

celda 29

In [15]:
#celda 30

csv_pob = rec_pob[rec_pob["format"] == "CSV"].copy()
csv_pob["any"] = csv_pob["name"].str.extract(r"(\d{4})").astype(int)

# Nos quedamos con nuestra ventana de análisis
csv_pob = csv_pob[(csv_pob["any"] >= 2010) & (csv_pob["any"] <= 2024)]
csv_pob = csv_pob.sort_values("any").reset_index(drop=True)

print("Años seleccionados:", csv_pob["any"].tolist())
print("\nReparto API (True) / CSV directo (False):")
print(csv_pob[["any", "datastore_active"]].to_string(index=False))

Años seleccionados: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Reparto API (True) / CSV directo (False):
 any  datastore_active
2010              True
2011              True
2012              True
2013              True
2014              True
2015              True
2016              True
2017              True
2018              True
2019              True
2020              True
2021              True
2022              True
2023              True
2024              True


In [16]:
#celda 31

def descargar_recurso(fila):
    """Descarga un recurso por API si está en el datastore, o por CSV directo si no."""
    if fila.datastore_active:
        df = fetch_datastore(fila.id)
    else:
        df = pd.read_csv(fila.url)
    df["any_recurso"] = fila.any
    return df

trozos_pob = []
for fila in csv_pob.itertuples():
    df_anyo = descargar_recurso(fila)
    trozos_pob.append(df_anyo)
    print(f"  {fila.any}: {len(df_anyo)} filas")

poblacion = pd.concat(trozos_pob, ignore_index=True)
print("\nTotal filas:", len(poblacion))
print("Columnas:", list(poblacion.columns))
poblacion.head()

  2010: 1068 filas
  2011: 1068 filas
  2012: 1068 filas
  2013: 1068 filas
  2014: 1068 filas
  2015: 1068 filas
  2016: 1068 filas
  2017: 1068 filas
  2018: 1068 filas
  2019: 1068 filas
  2020: 1068 filas
  2021: 1068 filas
  2022: 1068 filas
  2023: 1068 filas
  2024: 1068 filas

Total filas: 16020
Columnas: ['Codi_Districte', 'Nom_Districte', 'Codi_Barri', 'Nom_Barri', 'AEB', 'Seccio_Censal', 'Valor', 'Data_Referencia', '_id', 'any_recurso']


,Codi_Districte,Nom_Districte,Codi_Barri,Nom_Barri,AEB,Seccio_Censal,Valor,Data_Referencia,_id,any_recurso
0,1,Ciutat Vella,1,el Raval,1,1001,1327,2010-01-01T00:00:00,1,2010
1,1,Ciutat Vella,1,el Raval,1,1002,1483,2010-01-01T00:00:00,2,2010
2,1,Ciutat Vella,1,el Raval,2,1003,3266,2010-01-01T00:00:00,3,2010
3,1,Ciutat Vella,1,el Raval,2,1004,2917,2010-01-01T00:00:00,4,2010
4,1,Ciutat Vella,1,el Raval,3,1005,2401,2010-01-01T00:00:00,5,2010


### Guardar población y descargar edad y origen

Guardamos población cruda. Como edad y origen siguen el mismo patrón
(`AAAA_<dataset>.csv`), creamos una función `extraer_pad` que hace todo el ciclo
(explorar → filtrar 2010–2024 → descargar → guardar raw) y la reutilizamos.

celda 32

In [17]:
#celda 33

def extraer_pad(dataset_id, archivo_raw, y0=2010, y1=2024):
    """Ciclo completo para un dataset del padrón: explora, filtra años, descarga y guarda raw."""
    pkg = get_action("package_show", {"id": dataset_id})
    rec = pd.DataFrame(pkg["resources"])
    rec = rec[rec["format"] == "CSV"].copy()
    rec["any"] = rec["name"].str.extract(r"(\d{4})").astype(int)
    rec = rec[(rec["any"] >= y0) & (rec["any"] <= y1)].sort_values("any")

    trozos = [descargar_recurso(fila) for fila in rec.itertuples()]
    out = pd.concat(trozos, ignore_index=True)

    out.to_csv(RAW / archivo_raw, index=False, encoding="utf-8")
    print(f"{dataset_id}: {len(out)} filas, {len(out.columns)} columnas -> {archivo_raw}")
    print("   columnas:", list(out.columns), "\n")
    return out

# Edad (grupos quinquenales) y origen (nacionalidad Spain/EU/Rest of foreign)
edad   = extraer_pad("pad_mdbas_edat-q",            "padron_edad_2010_2024_raw.csv")
origen = extraer_pad("pad_mdbas_nacionalitat-g_sexe", "padron_origen_2010_2024_raw.csv")

pad_mdbas_edat-q: 326762 filas, 11 columnas -> padron_edad_2010_2024_raw.csv
   columnas: ['Codi_Districte', 'Nom_Districte', 'Codi_Barri', 'Nom_Barri', 'AEB', 'Seccio_Censal', 'Valor', 'Data_Referencia', '_id', 'EDAT_Q', 'any_recurso'] 

pad_mdbas_nacionalitat-g_sexe: 98827 filas, 12 columnas -> padron_origen_2010_2024_raw.csv
   columnas: ['Codi_Districte', 'SEXE', 'NACIONALITAT_G', 'Nom_Districte', 'Codi_Barri', 'Nom_Barri', 'AEB', 'Seccio_Censal', 'Valor', 'Data_Referencia', '_id', 'any_recurso'] 



In [ ]:
#celda 35.5 (el documento "padron_poblacion_2010_2024_raw.csv" no se habia descargado antes.)

ruta_pob = RAW / "padron_poblacion_2010_2024_raw.csv"
poblacion.to_csv(ruta_pob, index=False, encoding="utf-8")
print("Guardado:", ruta_pob.resolve())
print("Filas:", len(poblacion), "| Columnas:", len(poblacion.columns))

Guardado: C:\Users\User\Desktop\proyecto_bibliotecas_bcn\data\raw\padron_poblacion_2010_2024_raw.csv
Filas: 16020 | Columnas: 10


## Cierre de la Fase 1 — inventario de datos crudos

Listamos los ficheros guardados en `data/raw` con su número de filas y columnas,
para verificar que la extracción está completa antes de pasar a la Fase 2 (limpieza
y transformación).

celda 34

In [20]:
print("Archivos en data/raw:\n")
total = 0
for f in sorted(RAW.glob("*.csv")):
    df = pd.read_csv(f)
    total += len(df)
    print(f"  {f.name:48s} {len(df):>8,} filas  {len(df.columns):>2} cols")
print(f"\nTotal de ficheros: {len(list(RAW.glob('*.csv')))} | Filas totales: {total:,}")

Archivos en data/raw:

  bibliotecas_xarxa_2010_2022_raw.csv                 3,720 filas  20 cols
  bibliotecas_xarxa_2023_2024_raw.csv                   800 filas  17 cols
  padron_edad_2010_2024_raw.csv                     326,762 filas  11 cols
  padron_origen_2010_2024_raw.csv                    98,827 filas  12 cols
  padron_poblacion_2010_2024_raw.csv                 16,020 filas  10 cols
  renta_disponible_2015_2022_raw.csv                  8,544 filas   9 cols

Total de ficheros: 6 | Filas totales: 454,673
